# Algorithms for Approximate Reducts

## Iterative Attribute Selection

---

Workshop presentation

Practical aspects of searching approximate reducts using the `skrough` package

## Agenda

- Decision tables, indiscernibility, and approximate reducts (recap)
- What does a reduct look like? (Golf dataset example)
- Why it matters: classification gain and feature importance
- What takes long: profiling the bottleneck
- GroupIndex: the core data structure and its API
- GroupIndex implementations: from lazy to numba
- Benchmarks and end-to-end experiments
- (Bonus) Hook architecture for building algorithms

## Decision Tables

A decision table is a tuple $(U, A \cup \{d\})$:

- $U$ -- universe of objects
- $A$ -- conditional attributes
- $d$ -- decision attribute

**Example: Play Golf**

| Outlook | Temperature | Humidity | Wind | Play |
|---------|-------------|----------|------|------|
| sunny | hot | high | weak | no |
| sunny | hot | high | strong | no |
| overcast | hot | high | weak | yes |
| rain | mild | high | weak | yes |
| ... | ... | ... | ... | ... |

$|A| = 4$, $|U| = 14$, $V_d = \{\text{yes}, \text{no}\}$

## Indiscernibility & Disorder Measures

Given $B \subseteq A$, objects $u, v \in U$ are **indiscernible** iff they have the same
values on all attributes in $B$:

$$u \, IND(B) \, v \iff \forall a \in B: a(u) = a(v)$$

Each $B$ partitions $U$ into **equivalence classes** (groups).

A **disorder measure** quantifies how impure the decision values are within groups:

- **Entropy**: $-\sum p_i \log_2 p_i$
- **Gini impurity**: $1 - \sum p_i^2$
- **Conflicts count**: number of conflicting object pairs

Goal: find $B$ that makes groups as pure as possible (low disorder).

## Approximate Reducts

Given a decision table and $\varepsilon \in [0, 1)$:

- **Base disorder**: disorder with all objects in one group (no attributes)
- **Total disorder**: disorder with all attributes $A$
- **Threshold**: $T = total + \varepsilon \cdot (base - total)$

A subset $B \subseteq A$ is an **$\varepsilon$-approximate decision reduct** iff:

$$disorder\_score(B) \leq T$$

and no proper subset of $B$ satisfies this.

Intuition: $\varepsilon = 0$ means a full reduct (max quality).
$\varepsilon > 0$ means we accept some loss for fewer attributes.

## What Does a Reduct Look Like?

Play Golf dataset, $\varepsilon = 0.2$, entropy:

```
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Wind', 'Temperature']
...
```

9 out of 10 reducts pick the same 3 attributes.

Temperature is redundant -- Outlook + Humidity + Wind already achieve the threshold.

In [ ]:
import numpy as np
import pandas as pd

from skrough.dataprep import prepare_factorized_data
from skrough.disorder_measures import entropy
from skrough.algorithms.reducts import get_approx_reduct_greedy_heuristic

In [ ]:
df = pd.DataFrame(
    np.array(
        [
            ["sunny", "hot", "high", "weak", "no"],
            ["sunny", "hot", "high", "strong", "no"],
            ["overcast", "hot", "high", "weak", "yes"],
            ["rain", "mild", "high", "weak", "yes"],
            ["rain", "cool", "normal", "weak", "yes"],
            ["rain", "cool", "normal", "strong", "no"],
            ["overcast", "cool", "normal", "strong", "yes"],
            ["sunny", "mild", "high", "weak", "no"],
            ["sunny", "cool", "normal", "weak", "yes"],
            ["rain", "mild", "normal", "weak", "yes"],
            ["sunny", "mild", "normal", "strong", "yes"],
            ["overcast", "mild", "high", "strong", "yes"],
            ["overcast", "hot", "normal", "weak", "yes"],
            ["rain", "mild", "high", "strong", "no"],
        ],
        dtype=object,
    ),
    columns=["Outlook", "Temperature", "Humidity", "Wind", "Play"],
)
TARGET_COLUMN = "Play"
x, x_counts, y, y_count = prepare_factorized_data(df, TARGET_COLUMN)
df

In [ ]:
result = get_approx_reduct_greedy_heuristic(
    x=x,
    y=y,
    disorder_fun=entropy,
    epsilon=0.2,
    n_reducts=10,
)
for approx_reduct in result:
    print([df.columns[i] for i in approx_reduct.attrs])

## Why It Matters

Reducts are not just theoretical constructs -- they have practical value:

1. **Feature selection**: reducts identify minimal informative attribute subsets
2. **Classification**: ensembles of reducts can match or exceed standard classifiers
3. **Feature importance**: frequency and gain across reducts rank attributes

**Seismic dataset example** (133150 objects, 738 attrs):

| Method | BAC | ACC |
|--------|-----|-----|
| Decision Tree | 0.834 | 0.985 |
| XGBoost | 0.824 | 0.990 |
| Greedy reducts ensemble | **0.844** | 0.990 |
| VotingClassifier (reduct triplets) | **0.858** | **0.992** |

## What Takes Long?

Profiling the greedy reduct computation reveals the bottleneck:

![snakeviz lazy](code/profile_output/greedy_heuristic_lazy.png)

GroupIndex operations dominate -- split, compress, get_disorder_score.

The greedy loop evaluates many candidates per iteration, each calling split.

## GroupIndex in Context

In the greedy algorithm, each iteration needs `disorder_score(B U {a})` for all candidates `a`.

```python
gi = GroupIndex.create_uniform(n_objects)  # all objects in one group
for attr in selected_attrs:
    gi = gi.split(x[:, attr], x_counts[attr])  # refine by new attribute
gi.compress()  # renumber groups to be contiguous
score = gi.get_disorder_score(y, y_count, fun)  # evaluate impurity
```

**Key API:**

| Method | What it does |
|--------|-------------|
| `create_uniform(n)` | All objects in one group |
| `split(values, values_count)` | Refine groups by a new attribute |
| `compress()` | Renumber groups to be contiguous |
| `get_disorder_score(y, y_count, fun)` | Compute weighted disorder |
| `from_data(x, x_counts, attrs)` | Build from scratch (alternative) |

In [ ]:
from skrough.structs.group_index import GroupIndex

gi = GroupIndex.create_uniform(len(df))
print(f"Start: {gi.n_groups} group(s)")

for attr in [0, 2]:
    gi = gi.split(x[:, attr], x_counts[attr])
    print(f"After attr {attr}: {gi.n_groups} group(s)")

gi.compress()
print(f"Compressed: {gi.n_groups} group(s)")

## GroupIndex: Approach Categories

Different implementations trade off simplicity, memory, and speed:

1. **Lazy** (stateless) -- hash-based, vectorized string concat, sequential IDs.
   Minimal state, computes on demand.

2. **Dict** (stateful, forward index) -- dict-based groups, on-the-fly compactification.
   Maintains explicit group -> objects mapping.

3. **Hash** (stateful) -- xxhash-based, streaming disorder computation.
   Avoids full group materialization.

4. **Pure** (stateful) -- pure numpy, no numba dependency.
   Sort-based or algebraic, portable.

5. **Numba** (stateful, forward index) -- numba-accelerated, the default.
   JIT-compiled split and disorder computation.

## GroupIndex Benchmarks

![benchmark plot](code/group_index_benchmark_plot.png)

Seismic dataset, greedy heuristic: 30 candidates, 20 reducts, n_jobs=-1.

Each line adds one implementation -- progressive comparison.

Timeout: 3 minutes per run (skips larger sizes if exceeded).

## Numba: The Fastest

![snakeviz numba](code/profile_output/greedy_heuristic_numba.png)

Compared to lazy: numba runs on the full dataset and is significantly faster.

JIT compilation of split and disorder computation eliminates Python overhead.

## End-to-End: Seismic Dataset

133150 objects, 738 attributes, gini impurity, epsilon=0.05:

- **Greedy**: 100 reducts, avg length 16.38, top attr: `latest_seismic_assessment` (93/100)
- **DAAR**: 100 reducts, avg length 11.89, shorter but lower BAC (0.727 vs 0.844)

Feature importance from reducts:

| Attribute | Count | Global Gain |
|-----------|-------|-------------|
| latest_seismic_assessment | 93 | 0.318 |
| latest_progress_estimation_l | 63 | 0.081 |
| latest_progress_estimation_r | 51 | 0.058 |

## (Bonus) Hook Architecture

```
ProcessingMultiStage
  ├── init hooks:      prepare data, group index, threshold
  ├── stages
  │    └── Stage
  │         ├── stop_hooks:          when to stop?
  │         ├── pre_candidates_hooks: what to consider?
  │         ├── select_hooks:         which one to pick?
  │         ├── inner_process_hooks:  what to do with it?
  │         └── finalize_hooks:       cleanup
  └── prepare_result:   format output
```

Same pipeline, different hooks = different algorithms.

Greedy uses approximation-threshold stop + best-gain select;
DAAR uses the same pipeline but with a stochastic acceptance test.

## Summary

- **Approximate reducts** identify minimal informative attribute subsets
- **GroupIndex** is the core data structure -- its implementation matters
- **Numba** gives the best end-to-end performance on large datasets
- **Reducts as feature selection** can improve classifier performance
- **Hook architecture** enables building different algorithms from the same pipeline

## References

- `skrough` package: `src/skrough/`
- Theoretical foundations: `knowledgebase/`
- Example notebooks: `examples/`
- Benchmark code: `presentation/code/`

## Thank You

Questions?